# * TOL : Gross Adds & Subbase

In [7]:
import configparser
import datetime as dt
import pandas as pd
import numpy as np
import xlrd
import oracledb
import re
from functools import reduce

config = configparser.ConfigParser()
config.read('../../my_config.ini')
config.sections()

TDMDBPR_user = config['TDMDBPR']['username']
TDMDBPR_pwd = config['TDMDBPR']['password']
TDMDBPR_db = config['TDMDBPR']['db']
TDMDBPR_host = config['TDMDBPR']['host']
TDMDBPR_port = config['TDMDBPR']['port']

AKPIPRD_user = config['AKPIPRD']['username']
AKPIPRD_pwd = config['AKPIPRD']['password']
AKPIPRD_db = config['AKPIPRD']['db']
AKPIPRD_host = config['AKPIPRD']['host']
AKPIPRD_port = config['AKPIPRD']['port']

curr_dt = dt.datetime.now().date()
str_curr_dt = curr_dt.strftime('%Y%m%d')

In [8]:
''' Input parameter '''

op_dir = 'C:/Ruz/MyProject/Code/Jupyter/data/Monthly/output'
op_file = f'tol_ga_subs_monthly_{str_curr_dt}'

print(f'\nParameter input...\n')
print(f'   -> op_dir: {op_dir}')
print(f'   -> op_vis_file: {op_file}')


Parameter input...

   -> op_dir: C:/Ruz/MyProject/Code/Jupyter/data/Monthly/output
   -> op_vis_file: tol_ga_subs_monthly_20260309


## Step 1 : Import Data Source
-> AGG_PERF_NEWCO

In [9]:
''' Execute transaction '''


# Input parameter
v_as_date = 20260228
print(f'\nParameter input...')
print(f'   -> v_as_date: {v_as_date}')

curr_datetime = dt.datetime.now().strftime('%Y-%m-%d, %H:%M:%S')
print(f'\nData as of {curr_datetime}')


# Connect : TDMDBPR
src_dsn = f'{TDMDBPR_user}/{TDMDBPR_pwd}@{TDMDBPR_host}:{TDMDBPR_port}/{TDMDBPR_db}'
src_conn = oracledb.connect(src_dsn)
src_cur = src_conn.cursor()
query = (f"""
    WITH W_VAR AS 
    ( 
        SELECT V_YYYYMMDD, SUBSTR(V_YYYYMMDD,1,6) AS V_YYYYMM
        FROM (SELECT {v_as_date} AS V_YYYYMMDD FROM DUAL) TMP
    ) 

    SELECT /*+PARALLEL(8)*/ *
    FROM (	
        -->> Active Subs (VIS)
        SELECT TM_KEY_MTH, METRIC_CD, METRIC_NAME, AREA_TYPE, AREA_CD, AREA_NAME
            , ACTUAL_AGG_MTH AS ACTUAL, TARGET_AGG_MTH AS TARGET, PPN_TM
        FROM GEOSPCAPPO.AGG_PERF_NEWCO
        WHERE METRIC_CD = 'TB3S020604' --FTTx SubBase
        AND AREA_TYPE IN ('P', 'G', 'H', 'HH')
        AND TM_KEY_DAY = (SELECT V_YYYYMMDD FROM W_VAR)
        
        UNION ALL 
        
        -->> Gross Adds (VIS)
        SELECT TM_KEY_MTH, METRIC_CD, METRIC_NAME, AREA_TYPE, AREA_CD, AREA_NAME
            , ACTUAL_AGG_MTH AS ACTUAL, TARGET_AGG_MTH AS TARGET, PPN_TM
        FROM GEOSPCAPPO.AGG_PERF_NEWCO
        WHERE METRIC_CD IN (
            'TB3S000102CG' --TOL Gross Adds Connected : Consumer - GEO Channel
            , 'TB3S000102D1CG') --TOL Gross Adds Connected : Consumer - GEO Channel (Install Location)
        AND AREA_TYPE IN ('P', 'G', 'H', 'HH')
        AND TM_KEY_DAY = (SELECT V_YYYYMMDD FROM W_VAR)
        
    --	UNION ALL 
    --	
    --	SELECT TM_KEY_MTH, METRIC_CD, METRIC_NAME, AREA_TYPE, AREA_CD, AREA_NAME
    --		, ACTUAL_AGG_MTH AS ACTUAL, TARGET_AGG_MTH AS TARGET, PPN_TM
    --	FROM GEOSPCAPPO.AGG_PERF_NEWCO_CCAA
    --	WHERE METRIC_CD IN (
    --		'TB3S000102CG' --TOL Gross Adds Connected : Consumer - GEO Channel
    --		, 'TB3S000102D1CG') --TOL Gross Adds Connected : Consumer - GEO Channel (Install Location)
    --	AND AREA_TYPE = 'CCAA'
    --	AND TM_KEY_DAY = (SELECT V_YYYYMMDD FROM W_VAR)
        
        UNION ALL 
        
        -->> Gross Adds (CDS)
        SELECT CAST(SUBSTR(TM_KEY_DAY,1,6) AS INT) AS TM_KEY_MTH
            , METRIC_CD, METRIC_NAME, AREA_TYPE, AREA_CD, AREA_DESC
            , SUM(METRIC_VALUE) AS ACTUAL, NULL AS TARGET
            , MAX(LOAD_DATE) LOAD_DATE
        FROM CDSAPPO.DIM_CORP_KPI 
        WHERE METRIC_CD IN (
            'TB3S000102CG' --TOL Gross Adds Connected : Consumer - GEO Channel
            , 'TB3S000102D1CG' --TOL Gross Adds Connected : Consumer - GEO Channel (Install Location)
            )
        AND AREA_TYPE = 'CCAA'
        AND TM_KEY_DAY LIKE (SELECT V_YYYYMM FROM W_VAR)||'%'
        GROUP BY SUBSTR(TM_KEY_DAY,1,6), METRIC_CD, METRIC_NAME, AREA_TYPE, AREA_CD, AREA_DESC
    ) A
    --ORDER BY TM_KEY_MTH, METRIC_CD, AREA_TYPE, AREA_CD
""")


try:
    # Create Dataframe
    src_cur.execute(query)
    rows = src_cur.fetchall()
    src_df = pd.DataFrame.from_records(rows, columns=[x[0] for x in src_cur.description])
    print(f'\nDataFrame: {src_df.shape[0]} rows, {src_df.shape[1]} columns')
    
    src_cur.close()


except oracledb.DatabaseError as e:
    print(f'\nError with Oracle : {e}')


finally:
    src_conn.close()


Parameter input...
   -> v_as_date: 20260228

Data as of 2026-03-09, 16:54:51

DataFrame: 1746 rows, 9 columns


In [10]:
''' Summary '''

src_summary_df = src_df.copy()
src_summary_df = src_summary_df.groupby(['TM_KEY_MTH', 'METRIC_CD', 'METRIC_NAME', 'AREA_TYPE']).agg({'AREA_CD': 'count', 'ACTUAL': 'sum', 'TARGET': 'sum', 'PPN_TM': 'max'}).sort_values(by=['TM_KEY_MTH', 'METRIC_CD', 'AREA_TYPE']).reset_index()
# # src_summary_df = src_summary_df.astype({'ACTUAL' : int, 'TARGET': int})

mod_col_list = src_summary_df.iloc[:, 5:7].columns.tolist()
mod_col_list
for col in mod_col_list:
    src_summary_df[col] = src_summary_df[col].apply(lambda x: format(x, ',.0f'))

src_summary_df

,TM_KEY_MTH,METRIC_CD,METRIC_NAME,AREA_TYPE,AREA_CD,ACTUAL,TARGET,PPN_TM
0,202602,TB3S000102CG,TOL Gross Adds Connected : Consumer - GEO Channel,CCAA,334,"36,040",0,2026-03-09 16:48:26
1,202602,TB3S000102CG,TOL Gross Adds Connected : Consumer - GEO Channel,G,8,"36,040","44,072",2026-03-09 11:07:29
2,202602,TB3S000102CG,TOL Gross Adds Connected : Consumer - GEO Channel,H,64,"36,040","43,975",2026-03-09 02:46:14
3,202602,TB3S000102CG,TOL Gross Adds Connected : Consumer - GEO Channel,HH,96,"36,040","43,788",2026-03-09 03:54:58
4,202602,TB3S000102CG,TOL Gross Adds Connected : Consumer - GEO Channel,P,1,"36,040","44,044",2026-03-09 11:07:29
5,202602,TB3S000102D1CG,TOL Gross Adds Connected : Consumer - GEO Chan...,CCAA,905,"35,975",0,2026-03-09 16:48:26
6,202602,TB3S000102D1CG,TOL Gross Adds Connected : Consumer - GEO Chan...,G,8,"35,975",0,2026-03-09 11:07:29
7,202602,TB3S000102D1CG,TOL Gross Adds Connected : Consumer - GEO Chan...,H,64,"35,975",0,2026-03-09 02:46:14
8,202602,TB3S000102D1CG,TOL Gross Adds Connected : Consumer - GEO Chan...,HH,96,"35,975",0,2026-03-09 03:54:58
9,202602,TB3S000102D1CG,TOL Gross Adds Connected : Consumer - GEO Chan...,P,1,"36,040",0,2026-03-09 11:07:29


## Step 2 : Areas Mapping

In [14]:
''' Org Mapping '''

org_file = 'C:/Ruz/MyProject/Code/Jupyter/data/Monthly/data/Org_Mapping.xlsx'
org_sheet = 'data'
org_df = pd.read_excel(org_file, sheet_name=org_sheet)

org_df.head(3)

,AREA_TYPE,AREA_CD,AREA_NAME
0,P,P,PRESIDENT
1,G,GX1,Regional Management 1 (BMA-West)
2,G,GX2,Regional Management 2 (BMA-East)


In [15]:
''' Transaction '''

txn_df = src_df.copy()
txn_df = txn_df.drop('AREA_NAME', axis=1)

txn_df.head(3)

,TM_KEY_MTH,METRIC_CD,METRIC_NAME,AREA_TYPE,AREA_CD,ACTUAL,TARGET,PPN_TM
0,202602,TB3S020604,FTTx SubBase,H,003,77783,NaN,2026-03-09 02:46:14
1,202602,TB3S020604,FTTx SubBase,H,004,104662,NaN,2026-03-09 02:46:14
2,202602,TB3S020604,FTTx SubBase,H,005,90210,NaN,2026-03-09 02:46:14


In [16]:
''' Aggregate '''

txn_df = txn_df.query("AREA_TYPE in ('P', 'G', 'H', 'CCAA')").copy()
area_no_mapping = {'P': 1, 'G': 2, 'H': 3, 'CCAA': 4}
txn_df['AREA_NO'] = txn_df['AREA_TYPE'].map(area_no_mapping)
txn_df['%ACH'] = (txn_df['ACTUAL']/txn_df['TARGET'])*100

txn_df.head(3)

,TM_KEY_MTH,METRIC_CD,METRIC_NAME,AREA_TYPE,AREA_CD,ACTUAL,TARGET,PPN_TM,AREA_NO,%ACH
0,202602,TB3S020604,FTTx SubBase,H,003,77783,NaN,2026-03-09 02:46:14,3,NaN
1,202602,TB3S020604,FTTx SubBase,H,004,104662,NaN,2026-03-09 02:46:14,3,NaN
2,202602,TB3S020604,FTTx SubBase,H,005,90210,NaN,2026-03-09 02:46:14,3,NaN


In [17]:
''' Last Results '''

measure_cols = ['AREA_CD', 'ACTUAL', 'TARGET', '%ACH']

# FTTx SubBase
subs_df = txn_df.query("METRIC_CD=='TB3S020604' and AREA_TYPE in ('P', 'G', 'H')") 
subs_df = subs_df[measure_cols]
subs_df = subs_df.rename(columns={'ACTUAL': 'ACTUAL_SUBS','TARGET': 'TARGET_SUBS','%ACH': '%ACH_SUBS'})

# TOL Gross Adds Connected : Consumer - GEO Channel
ga_sale_df = txn_df.query("METRIC_CD=='TB3S000102CG' and AREA_TYPE in ('P', 'G', 'H', 'CCAA')") 
ga_sale_df = ga_sale_df[measure_cols]
ga_sale_df = ga_sale_df.rename(columns={'ACTUAL': 'ACTUAL_GA','TARGET': 'TARGET_GA','%ACH': '%ACH_GA'})

# TOL Gross Adds Connected : Consumer - GEO Channel (Install Location)
ga_install_df = txn_df.query("METRIC_CD=='TB3S000102D1CG' and AREA_TYPE in ('P', 'G', 'H', 'CCAA')") 
ga_install_df = ga_install_df[measure_cols]
ga_install_df = ga_install_df.rename(columns={'ACTUAL': 'ACTUAL_GA_INSTALL','TARGET': 'TARGET_GA_INSTALL','%ACH': '%ACH_GA_INSTALL'})

result_df = pd.merge(org_df, subs_df, how='left', on='AREA_CD')
result_df = pd.merge(result_df, ga_sale_df, how='left', on='AREA_CD')
result_df = pd.merge(result_df, ga_install_df, how='left', on='AREA_CD')
result_df

,AREA_TYPE,AREA_CD,AREA_NAME,ACTUAL_SUBS,TARGET_SUBS,%ACH_SUBS,ACTUAL_GA,TARGET_GA,%ACH_GA,ACTUAL_GA_INSTALL,TARGET_GA_INSTALL,%ACH_GA_INSTALL
0,P,P,PRESIDENT,3355843.0,NaN,NaN,36040.0,44044.0,81.827264,36040.0,NaN,NaN
1,G,GX1,Regional Management 1 (BMA-West),580199.0,NaN,NaN,5983.0,6244.0,95.819987,3613.0,NaN,NaN
2,G,GX2,Regional Management 2 (BMA-East),865969.0,NaN,NaN,7200.0,7728.0,93.167702,7632.0,NaN,NaN
3,G,GX3,Regional Management 3 (East),298252.0,NaN,NaN,3611.0,5124.0,70.472287,3934.0,NaN,NaN
4,G,GX4,Regional Management 4 (North),369579.0,NaN,NaN,3759.0,5068.0,74.171271,4112.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
995,CCAA,9609,NARATHIWAT_SUKHIRIN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,NaN,NaN
996,CCAA,9610,NARATHIWAT_SU-NGAI KOLOK,NaN,NaN,NaN,NaN,NaN,NaN,22.0,NaN,NaN
997,CCAA,9611,NARATHIWAT_SU-NGAI PADI,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN
998,CCAA,9612,NARATHIWAT_CHANAE,NaN,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN


## Step 3 : Output Files

In [18]:
''' Generate CSV file '''

result_df.to_csv(f'{op_dir}/{op_file}.csv', index=False, encoding='utf-8')
print(f'\n   -> Generate "{op_file}.csv" successfully')


   -> Generate "tol_ga_subs_monthly_20260309.csv" successfully


## Review

In [19]:
result_df

,AREA_TYPE,AREA_CD,AREA_NAME,ACTUAL_SUBS,TARGET_SUBS,%ACH_SUBS,ACTUAL_GA,TARGET_GA,%ACH_GA,ACTUAL_GA_INSTALL,TARGET_GA_INSTALL,%ACH_GA_INSTALL
0,P,P,PRESIDENT,3355843.0,NaN,NaN,36040.0,44044.0,81.827264,36040.0,NaN,NaN
1,G,GX1,Regional Management 1 (BMA-West),580199.0,NaN,NaN,5983.0,6244.0,95.819987,3613.0,NaN,NaN
2,G,GX2,Regional Management 2 (BMA-East),865969.0,NaN,NaN,7200.0,7728.0,93.167702,7632.0,NaN,NaN
3,G,GX3,Regional Management 3 (East),298252.0,NaN,NaN,3611.0,5124.0,70.472287,3934.0,NaN,NaN
4,G,GX4,Regional Management 4 (North),369579.0,NaN,NaN,3759.0,5068.0,74.171271,4112.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
995,CCAA,9609,NARATHIWAT_SUKHIRIN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,NaN,NaN
996,CCAA,9610,NARATHIWAT_SU-NGAI KOLOK,NaN,NaN,NaN,NaN,NaN,NaN,22.0,NaN,NaN
997,CCAA,9611,NARATHIWAT_SU-NGAI PADI,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN
998,CCAA,9612,NARATHIWAT_CHANAE,NaN,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN
